# Notebook 4: Ask the twin

The twin of the hydrogen chain computes one night in steps of one minute. This notebook asks it questions: first
one, then the same one at many times of day. The inputs come from the series of the signals of the system (wind of
the station, price) where they exist; every answer says where its inputs come from and how far it can be relied on.

A run is computed on the server and kept there in memory. This notebook writes nothing to the broker.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)
print("Retrieval server:", RETRIEVAL)

## 1. What can be asked

The scenario names its controls, the operating policies of the electrolyser and the decisions of its night.

In [ ]:
sc = client.scenario(SYSTEM)
print(sc["title"])
print(sc["decision"] or sc["description"])
print("Start", sc["start"], f"| {sc['steps']} steps of {sc['stepSeconds']} s")
print("\nDecisions of the night:")
for d in sc["referenceDecisions"]:
    print(f"  {d['clock'][:5]}  {d['atom']}.{d['control']} = {d['value']}  ({d['reason']})")
pd.DataFrame(sc["controls"])[["atom", "control", "default", "unit"]]

## 2. One question

What if the handover station opens its valve again at 02:00? Computed on the series of the signals and compared
with the decisions of the night on the same inputs.

In [ ]:
run = client.run(SYSTEM, [{"at": "02:00", "atom": "handover", "control": "valve", "value": 1}], inputs="signals")
print("Run", run["run"], "|", run["label"])
print("compared with", run["baseline"])
print("Status:", run["status"]["usability"])
for r in run["status"]["reasons"]:
    print("  -", r)
pd.DataFrame(run["inputs"])[["port", "source", "origin", "coverage"]]

In [ ]:
cmp = client.compare(SYSTEM, run["run"], run["baseline"])
kpis = pd.DataFrame(cmp["kpis"])[["label", "base", "run", "delta", "unit", "verdict"]]
kpis.columns = ["KPI", "reference", "this run", "difference", "unit", "verdict"]
kpis.round(1)

In [ ]:
start = pd.Timestamp(cmp["start"])
step = pd.Timedelta(seconds=cmp["stepSeconds"])
fig, axes = plt.subplots(1, len(cmp["series"]), figsize=(15, 3.5))
for ax, (key, s) in zip(np.atleast_1d(axes), cmp["series"].items()):
    idx = pd.date_range(start, periods=len(s["base"]), freq=step)
    ax.plot(idx, [np.nan if v is None else v for v in s["base"]], color="#8a949e", label="reference")
    ax.plot(idx, [np.nan if v is None else v for v in s["run"]], color="#1f6fb3", label="this run")
    for lim in s["limits"]:
        ax.axhline(lim["value"], color="#b3392b", ls="--", lw=0.8)
    ax.set_title(s["label"])
    ax.tick_params(axis="x", labelrotation=30)
np.atleast_1d(axes)[0].legend()
plt.tight_layout()
plt.show()

## 3. The same question at many times of day

By when must the valve be open again? One run per time of day, from 01:10 to 03:00 in steps of ten minutes. Every
row carries the id of its run.

In [ ]:
rows = []
for minute in range(70, 181, 10):
    at = f"{minute // 60:02d}:{minute % 60:02d}"
    r = client.run(SYSTEM, [{"at": at, "atom": "handover", "control": "valve", "value": 1}], inputs="signals")
    k = r["kpis"]
    rows.append({"valve open at": at, "protective trips": k.get("ely.trips"), "highest buffer pressure [bar]": k.get("buffer.p_max_bar"),
                 "delivered [kg]": k.get("handover.delivered_kg"), "not delivered [kg]": k.get("handover.shortfall_kg"), "run": r["run"]})
sweep = pd.DataFrame(rows)
sweep.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sweep["valve open at"], sweep["highest buffer pressure [bar]"], marker="o", label="highest buffer pressure")
ax.axhline(200, color="#b3392b", ls="--", lw=0.8, label="trip threshold")
ax.set_ylabel("bar")
ax.set_xlabel("valve open again at")
ax2 = ax.twinx()
ax2.bar(sweep["valve open at"], sweep["not delivered [kg]"], alpha=0.25, color="#c98a00", label="not delivered")
ax2.set_ylabel("kg")
ax.legend(loc="upper left")
ax.set_title("By when must the valve be open?")
plt.show()
latest = sweep[sweep["protective trips"] == 0]["valve open at"]
print("No protective trip up to:", latest.iloc[-1] if len(latest) else "none of the computed times")

## What these answers support

Every run is `restricted`, and the reasons are in its status: the demand of the offtaker is a demo generator, the
schedule and the grid meter are missing, the plant is assumed. The computation shows the mechanism of the chain,
not an effect that an operation could rely on. An offtake profile of the operator and the data of the plant
change that.